In [ ]:
!pip -q install f5-tts "pocket-tts @ git+https://github.com/mallahyari/pocket-tts@main" speechbrain faster-whisper jiwer pyloudnorm noisereduce num2words silero-vad onnxruntime 2>&1 | tail -5


In [ ]:
import hashlib
import time as _clock
NOTEBOOK_START = _clock.time()
import importlib.metadata
import json
import os
import platform
import shutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path


def captured(command):
    result = subprocess.run(command, capture_output=True, text=True)
    return result.returncode, (result.stdout + result.stderr).strip()


probe = subprocess.run([sys.executable, "-c", "from transformers import PreTrainedModel"],
                       text=True, capture_output=True)
if probe.returncode != 0 and "torchvision" in probe.stderr:
    print("transformers cannot import beside this torchvision - removing torchvision")
    print(captured([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchvision"])[1])
    probe = subprocess.run([sys.executable, "-c", "from transformers import PreTrainedModel"],
                           text=True, capture_output=True)
assert probe.returncode == 0, probe.stderr[-2000:]

import torch

OFFICIAL = {'torch': '2.8.0+cu128', 'torchaudio': '2.8.0+cu128', 'transformers': '5.3.0', 'accelerate': '1.13.0', 'webdataset': '1.0.2', 'safetensors': '0.7.0', 'tokenizers': '0.22.2', 'huggingface-hub': '1.7.2', 'tensorboardX': '2.6.4', 'soundfile': '0.13.1', 'librosa': '0.11.0', 'pydub': '0.25.1', 'numpy': '2.4.3', 'triton': '3.4.0'}
PINNED = ('torch', 'torchaudio', 'transformers', 'accelerate', 'webdataset', 'safetensors', 'tokenizers', 'huggingface-hub')


def installed(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


def matches(have, want):
    if have is None:
        return False
    return have == want or (have.split("+")[0] == want.split("+")[0] and "+" not in want)


stack = {name: {"official": want, "installed": installed(name),
                "matches": matches(installed(name), want), "pinned_here": name in PINNED}
         for name, want in OFFICIAL.items()}
direct = json.loads(importlib.metadata.distribution("omnivoice").read_text("direct_url.json") or "{}")
code, driver = captured(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"])
ENVIRONMENT_RECORD = {
    "python": platform.python_version(),
    "torch_cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "bf16_supported_including_emulation": torch.cuda.is_bf16_supported(),
    "gpus": [{"name": torch.cuda.get_device_name(i), "capability": list(torch.cuda.get_device_capability(i))}
             for i in range(torch.cuda.device_count())],
    "nvidia_smi": driver,
    "official_stack": stack,
    "omnivoice": {"version": installed("omnivoice"),
                  "commit": (direct.get("vcs_info") or {}).get("commit_id"), "url": direct.get("url")},
    "other_packages": {name: installed(name) for name in ("torchvision", "ipykernel", "papermill", "scipy")},
}
Path("/kaggle/working/environment.json").write_text(json.dumps(ENVIRONMENT_RECORD, indent=2))
for name, row in stack.items():
    mark = "ok" if row["matches"] else ("DIFFERS" if row["pinned_here"] else "differs - image's version kept")
    print(f"{name:16} official {row['official']:12} installed {str(row['installed']):16} {mark}")
print("omnivoice", ENVIRONMENT_RECORD["omnivoice"])
print("cuda", torch.version.cuda, "| cudnn", torch.backends.cudnn.version(), "|", driver)
for name in PINNED:
    assert stack[name]["matches"], f"{name} is not the official {OFFICIAL[name]}: {stack[name]['installed']}"
assert ENVIRONMENT_RECORD["omnivoice"]["commit"] == "08be0b4ccbac3e13e374e86fbfead4b4cac343e2", ENVIRONMENT_RECORD["omnivoice"]
assert torch.cuda.device_count() >= 2, "the official recipe trains on two GPUs - use T4 x2"
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"cuda:{i}: {torch.cuda.get_device_name(i)}, capability {torch.cuda.get_device_capability(i)}, "
          f"{free / 2**30:.1f} GiB free of {total / 2**30:.1f}")
print("disk:", f"{shutil.disk_usage('/tmp').free / 2**30:.0f} GiB free under /tmp")

INPUT = Path("/kaggle/input")
found = sorted(p for p in INPUT.rglob("repo/scripts/omnivoice_adapt.py") if "persian-voice-eval" in p.parts)
if not found:
    for path in sorted(p for p in INPUT.rglob("*") if p.is_dir())[:40]:
        print("   ", path)
    raise SystemExit("no repo with scripts/omnivoice_adapt.py under /kaggle/input")
REPO = Path("/kaggle/working/repo")
if REPO.exists():
    shutil.rmtree(REPO)
shutil.copytree(found[0].parent.parent, REPO)
# Kaggle unpacks the .gz when it builds the dataset; either form is checked
# against the hash the split builder recorded for its decompressed content.
MANIFESTS = REPO / "eval/manifests"
SPLITS = next((p for p in (MANIFESTS / "adaptation_splits_v1.jsonl.gz", MANIFESTS / "adaptation_splits_v1.jsonl")
               if p.is_file()), None)
assert SPLITS is not None, "the dataset predates the adaptation splits"
import gzip  # noqa: E402

content = gzip.open(SPLITS).read() if SPLITS.suffix == ".gz" else SPLITS.read_bytes()
expected = json.loads((MANIFESTS / "adaptation_splits_v1.summary.json").read_text())["records_sha256"]
assert hashlib.sha256(content).hexdigest() == expected, "the split file is not the one the builder wrote"
print("splits:", SPLITS.name, "sha256 matches the summary")

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONPATH"] = str(REPO / "src")
os.environ["VOICECLONE_SCORING_ENV"] = "kaggle:persian-voice-omnivoice-arms-v1"
sys.path.insert(0, str(REPO / "scripts"))
from kernel_gates import Gates, report_group, run_bounded  # noqa: E402

GATES = Gates()
RUNS = {}
LOGS = Path("/kaggle/working/logs")
LOGS.mkdir(exist_ok=True)
WORK = Path("/tmp/adapt")
WORK.mkdir(parents=True, exist_ok=True)
RECORDS = Path("/kaggle/working/records")
RECORDS.mkdir(exist_ok=True)


In [ ]:
from runner_check import check, diagnostics  # noqa: E402

ok, reason, run = check(run_bounded, Path("/kaggle/working/runner_check"), REPO / "scripts/moss15_worker.py")
RUNS["runner_check"] = run.record()
print(run.summary())
seen, line = diagnostics(report_group)
print("descriptor report:", "works" if seen else "DOES NOT WORK", "-", line)
GATES.record("runner", ok, reason)


In [ ]:
def fetch(plan, minutes, flags=()):
    """scripts/fetch_inputs.py under run_bounded: every request timed out, the stage bounded."""
    plan_path, report_path = RECORDS / "fetch_plan.json", RECORDS / "fetch.json"
    plan_path.write_text(json.dumps(plan))
    run = run_bounded([sys.executable, str(REPO / "scripts/fetch_inputs.py"), "--plan", str(plan_path),
                       "--report", str(report_path), *flags], timeout=minutes * 60, log=LOGS / "fetch.log",
                      monitor_every=300)
    RUNS["fetch"] = run.record()
    print(run.summary())
    report = json.loads(report_path.read_text()) if report_path.is_file() else {}
    fetched = {step["name"]: {k: step.get(k) for k in ("gib", "seconds", "tries", "failed_check")}
               for step in report.get("steps", [])}
    return run.ok and report.get("ok") is True, fetched

# Both engines run in this interpreter: nothing here is pinned to protect.
os.environ["VOICECLONE_POCKET_PYTHON"] = sys.executable
provenance = found[0].parents[2] / "PROVENANCE.md"
os.environ["VOICECLONE_CODE_COMMIT"] = (provenance.read_text().split("commit ")[-1].strip(" .\n")
                                        if provenance.is_file() else "the dataset's PROVENANCE.md was not found")
PARQUET = {"neyshekar": Path("/tmp/neyshekar-test"), "cv26": Path("/tmp/cv26")}
TEXTS = REPO / "eval/manifests/test_texts_v1.json"
NEEDED = ["scripts/prepare_test_corpus.py", "scripts/pocket_worker.py", "eval/manifests/test_texts_v1.json"]
missing = [name for name in NEEDED if not (REPO / name).is_file()]
if "pocket" not in (REPO / "src/voiceclone/engines/__init__.py").read_text():
    missing.append("the pocket engine")
GATES.record("code", not missing, f"missing from the dataset: {missing}" if missing else "the arms code is present")
if not GATES.skip("download", ("runner", "code")):
    fetched_ok, fetched = fetch({'snapshots': [{'name': 'neyshekar test', 'repo': 'shekar-ai/neyshekar', 'repo_type': 'dataset', 'revision': '30696ada746a66bb3047c984744b5ecfd60fe4e9', 'allow_patterns': ['data/test-*'], 'local_dir': '/tmp/neyshekar-test', 'forbid': ['train-*', 'validation-*']}, {'name': 'common voice 26', 'repo': 'pymmdrza/Common-Voice-Speech-26.0-Persian-Clean', 'repo_type': 'dataset', 'revision': '142e9bb40b56d9491faa0ec0987ee8d408bd9c05', 'allow_patterns': ['data/*'], 'local_dir': '/tmp/cv26'}, {'name': 'f5 persian', 'repo': 'Lumos675/F5_TTS_Persian', 'revision': 'cc322511e417', 'allow_patterns': ['model_last.pt', 'vocab.txt'], 'local_dir': '/tmp/f5-persian'}, {'name': 'f5 base', 'repo': 'SWivid/F5-TTS', 'revision': '84e5a410d9ce', 'allow_patterns': ['F5TTS_v1_Base/*'], 'local_dir': '/tmp/f5-base'}, {'name': 'pocket', 'repo': 'mehdi-hf/pocket-tts-farsi-v2', 'revision': 'e87097ed98edb8bd9c9bcabe9b575a51a466d0e4', 'allow_patterns': ['*.yaml', '*.safetensors', '*.py', '*.model', 'README.md'], 'local_dir': '/tmp/pocket/model'}, {'name': 'pocket g2p', 'repo': 'mehdi-hf/Homo-GE2PE-Persian-HF', 'revision': 'e314087bc761e323d127ee7899cb34f7c7d697cd', 'local_dir': '/tmp/pocket/g2p'}]}, 45, ['--no-xet'])
    GATES.record("download", fetched_ok, json.dumps(fetched))


In [ ]:
SETS = {'ney': ('neyshekar', 'primary', 28), 'cv': ('cv26', 'secondary', 30)}
CORPORA = {}
if not GATES.skip("corpora", ("download",)):
    ok = True
    for name, (source, role, planned) in SETS.items():
        out = Path(f"/tmp/test-{name}")
        run = run_bounded([sys.executable, str(REPO / "scripts/prepare_test_corpus.py"), "--splits", str(SPLITS),
                           "--source", source, "--parquet-dir", str(PARQUET[source]), "--out", str(out)],
                          timeout=2 * 3600, log=LOGS / f"corpus_{name}.log", monitor_every=600)
        RUNS[f"corpus_{name}"] = run.record()
        print(run.summary())
        if not run.ok or not (out / "roster.json").is_file():
            ok = False
            continue
        roster = json.loads((out / "roster.json").read_text())
        CORPORA[name] = {"dir": out, "roster": roster, "speakers": len(roster["speakers"]), "planned": planned}
        # Fewer than planned is reported, never topped up: no speaker is added after looking.
        print(name, len(roster["speakers"]), "speakers of", planned, "| replaced", roster["replaced"],
              "| left out", roster["left_out"])
        ok = ok and len(roster["speakers"]) >= 5
    GATES.record("corpora", ok, json.dumps({name: {k: v for k, v in c.items() if k != "dir"}
                                             for name, c in CORPORA.items()}, default=str))


In [ ]:
if not GATES.skip("configure", ("download",)):
    os.environ["VOICECLONE__ENGINES__F5__CHECKPOINT"] = "/tmp/f5-persian/model_last.pt"
    os.environ["VOICECLONE__ENGINES__F5__VOCAB"] = "/tmp/f5-persian/vocab.txt"
    os.environ["VOICECLONE__ENGINES__F5__FALLBACK_CHECKPOINT"] = (
        "/tmp/f5-base/F5TTS_v1_Base/model_1250000.safetensors")
    os.environ["VOICECLONE__ENGINES__POCKET__MODEL_DIR"] = "/tmp/pocket/model"
    os.environ["VOICECLONE__ENGINES__POCKET__G2P_DIR"] = "/tmp/pocket/g2p"
    present = {name: Path(path).exists() for name, path in (
        ("f5", "/tmp/f5-persian/model_last.pt"),
        ("f5 base", "/tmp/f5-base/F5TTS_v1_Base/model_1250000.safetensors"),
        ("pocket", "/tmp/pocket/model/model.safetensors"),
        ("g2p", "/tmp/pocket/g2p/config.json"))}
    GATES.record("configure", all(present.values()), json.dumps(present))


In [ ]:
SEEDS = [42, 43]
ARMS = ['f5', 'pocket']
LABELS = {}


def remaining_hours():
    return 11.25 - (time.time() - NOTEBOOK_START) / 3600


def bench(label, engine, corpus_dir, speakers, *extra):
    timeout = min(3.0, remaining_hours() - 0.5) * 3600
    if timeout < 1800:
        print(f"{label}: {remaining_hours():.2f} h left in the session - not started")
        return False, {}
    run = run_bounded([sys.executable, "scripts/benchmark.py", "--label", label, "--engine", engine,
                       "--corpus-dir", str(corpus_dir), "--sentences-file", str(TEXTS),
                       "--seeds", *map(str, SEEDS), "--device", "cuda", *extra],
                      cwd=REPO, timeout=timeout, log=LOGS / f"{label}.log", monitor_every=600)
    RUNS[label] = run.record()
    print(run.summary())
    results = REPO / "eval/results" / f"{label}.json"
    summary = json.loads(results.read_text())["summary"] if results.is_file() else {}
    expected = speakers * 12 * len(SEEDS)
    return run.ok and summary.get("failures", 1) == 0 and summary.get("renders", 0) == expected, summary


if not GATES.skip("renders", ("corpora", "configure")):
    facts = {}
    for name, (source, role, planned) in SETS.items():
        corpus = CORPORA[name]
        LABELS[name] = {"role": role}
        for arm in ARMS:
            label = f"{arm}-test-{name}"
            LABELS[name][arm] = label
            # The recogniser's floor on this set's real clips, measured once here.
            extra = ["--asr-floor"] if arm == ARMS[0] else []
            passed, summary = bench(label, arm, corpus["dir"], corpus["speakers"], *extra)
            facts[label] = {"held_out": summary.get("held_out_mean"), "cer": summary.get("cer_mean"),
                            "renders": summary.get("renders"), "complete": passed}
            print(label, json.dumps(facts[label]))
    # Every arm is secondary: one that fails is reported, and the session still
    # keeps what the others produced.
    GATES.record("renders", any(f["complete"] for f in facts.values()), json.dumps(facts))


In [ ]:
destination = Path("/kaggle/working/arms")
destination.mkdir(exist_ok=True)
for path in (REPO / "eval/results").glob("*.json"):
    shutil.copy2(path, destination / path.name)
(destination / "labels.json").write_text(json.dumps(LABELS, indent=2))
(destination / "gates.json").write_text(json.dumps(GATES.summary(), indent=2))
(destination / "runs.json").write_text(json.dumps(RUNS, indent=2))
shutil.copy2("/kaggle/working/environment.json", destination / "environment.json")
renders = Path("/kaggle/working/renders")
renders.mkdir(exist_ok=True)
for name, labels in LABELS.items():
    for key, label in labels.items():
        folder = REPO / "outputs/benchmark" / label if key != "role" else None
        for path in (folder.glob("*.wav") if folder and folder.is_dir() else []):
            shutil.copy2(path, renders / f"{label}__{path.name}")
print("gates:", json.dumps(GATES.summary(), indent=1))
